# 特徵融合

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/11-fusion/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本版使用 CPU 驗證機制及短步參數更新；長訓練的辨識效果、AP 與 GPU 效率仍待補測。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.1.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 本輪只驗證 CPU。已裝相同 PyTorch 版本時保留其 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
from torch import nn
import torch.nn.functional as F


class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.reduce = nn.Conv2d(16,8,1)
        self.mix = nn.Conv2d(16,8,3,padding=1)

    def forward(self,shallow,deep):
        reduced = self.reduce(deep)
        up = F.interpolate(reduced,size=shallow.shape[-2:],mode='nearest')
        return self.mix(torch.cat([shallow,up],dim=1))


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    shallow = torch.randn(1,8,8,8,requires_grad=True)
    deep = torch.randn(1,16,4,4,requires_grad=True)
    model = Fusion()
    optimizer = torch.optim.SGD(model.parameters(),lr=.01)
    before = model.mix.weight.detach().clone()
    output = model(shallow,deep)
    assert output.shape == (1,8,8,8)
    loss = output.square().mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    assert torch.isfinite(shallow.grad).all() and torch.isfinite(deep.grad).all()
    assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    assert shallow.grad.abs().sum() > 0 and deep.grad.abs().sum() > 0
    optimizer.step()
    assert not torch.equal(before,model.mix.weight.detach())
    # Different channels can legally concatenate; only spatial axes must match.
    unreduced = F.interpolate(deep.detach(),size=(8,8),mode='nearest')
    joined = torch.cat([shallow.detach(),unreduced],dim=1)
    assert joined.shape == (1,24,8,8)
    assert nn.Conv2d(24,8,3,padding=1)(joined).shape == (1,8,8,8)
    matched = model.reduce(deep.detach())
    matched = F.interpolate(matched,size=(8,8),mode='nearest')
    assert torch.cat([shallow.detach(),matched],dim=1).shape == (1,16,8,8)
    assert (shallow.detach()+matched).shape == (1,8,8,8)
    # A fully hand-checkable nearest-neighbor interpolation example.
    toy = torch.tensor([[[[1.,2.],[3.,4.]]]],requires_grad=True)
    enlarged = F.interpolate(toy,size=(4,4),mode='nearest')
    expected = torch.tensor([[[[1.,1.,2.,2.],[1.,1.,2.,2.],
                               [3.,3.,4.,4.],[3.,3.,4.,4.]]]])
    assert torch.equal(enlarged,expected)
    enlarged.sum().backward()
    assert torch.equal(toy.grad,torch.full_like(toy,4))
    print('nearest example',enlarged[0,0].tolist(),'source gradient',toy.grad[0,0].tolist())
    print('8ch shallow + 8ch upsampled deep -> 16ch concat -> 8ch mixed')
    print('parameters',sum(p.numel() for p in model.parameters()),'both branches backward and one step')


if __name__ == '__main__':
    main()


nearest example [[1.0, 1.0, 2.0, 2.0], [1.0, 1.0, 2.0, 2.0], [3.0, 3.0, 4.0, 4.0], [3.0, 3.0, 4.0, 4.0]] source gradient [[4.0, 4.0], [4.0, 4.0]]
8ch shallow + 8ch upsampled deep -> 16ch concat -> 8ch mixed
parameters 1296 both branches backward and one step
